# Topic 02 · Checking recent research on training: Adam and rare classes, the edge of stability, grokking

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd02-nghien-cuu-huan-luyen-en.ipynb)

This notebook accompanies [Topic 02 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/nghien-cuu-huan-luyen/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd02-nghien-cuu-huan-luyen.ipynb)):
the code, the data and every check are the same.

Three recent research results on how neural networks are trained, each checked again on a problem small enough to
run on a CPU. We don't take the papers' word for it: we set up the experiment again, run several seeds, and state
clearly whether the result **holds**, **does not hold** or **depends**.

**What you'll do:**
1. See why Adam beats gradient descent when the classes have very uneven frequencies (Kunstner et al., 2024).
2. Measure the largest curvature of the loss during training, and see that it stops right at $2/\eta$: "the edge of
   stability" (Cohen et al., 2021).
3. Reproduce grokking on addition modulo 113 and test the "softmax collapse" explanation (Prieto et al., 2025).

**Background:** [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/)
(a step is stable when $\eta < 2/\lambda$), [Lesson 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/)
and [Lesson 08 · Overfitting & regularization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chong-qua-khop/).
**Run time:** about 1 minute on a CPU, including the first MNIST download.

Like Topic 01, this notebook uses **PyTorch**. Each section runs a short version live, then loads the full results
(several seeds, several learning rates, longer runs) that the course's scripts `reel/research/*.py` computed in
advance. The live part uses exactly the same code and the same seeds as those scripts, so the notebook can check that
it gets the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

PyTorch, numpy and matplotlib (Colab has them preinstalled). We use 2 CPU threads, like a free Colab machine and like
the scripts: with a different number of threads, floating-point sums are added up in a different order, and the last
digits drift. `num()` prints numbers in the English format (a decimal point, commas between thousands); `sigfig()`
prints very small or very large numbers as $3.3 \cdot 10^{-9}$.

In [ ]:
import gzip
import hashlib
import json
import math
import os
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sigfig(x, sig=3):
    """sig significant digits: 0.243 · 0.0884 · 55.7 · 6.60·10⁻³"""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    if -2 <= e < 4:
        return num(x, max(0, sig - 1 - e))
    return f"{num(x / 10**e, sig - 1)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Data

Section 4 uses the first 1,000 MNIST images (we download the two training files and check their MD5). The scripts'
three result files are loaded with `fetch_json`, also with an MD5 check.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
}
DATA = Path("mnist")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


TAIL = fetch_json("research-adam-tail.json", "bf394b41a981e52d6208adbc82860f12")
EOS = fetch_json("research-eos.json", "190b3db8f62a01b99e46c6b9bf237c43")
GROK = fetch_json("research-grok.json", "6a50b726ed7d60635a3262c1a5da4621")

## 3 · Why Adam beats gradient descent on rare classes

**Claim** (Kunstner et al., 2024): when the class frequencies are very uneven — like the vocabulary of a language,
where a few words make up most of the text and tens of thousands of words are rare — gradient descent learns the
common classes quickly and the rare classes very slowly, while Adam (and sign descent, which steps along the sign of
the gradient) makes steady progress on every class. According to the authors, this is a large part of the reason
Adam outperforms SGD when training language models. Two later papers (Kunstner & Bach, 2025; Yadav et al., 2025)
prove this on small linear models.

We use the paper's own synthetic problem (Appendix A.2), at a small size: 7 **frequency groups**. Group 1 has 1 class
with 128 examples, group 2 has 2 classes with 64 examples each, …, group 7 has 64 classes with 2 examples each: the
frequency halves while the number of classes doubles, so each group has exactly 128 examples. In total, 896 examples
and 127 classes. The input is 1,024 random numbers drawn uniformly from $[0, 1]$, **completely unrelated to the
label**: there is nothing to "understand", but there are more dimensions than examples, so the linear model
$z = xW + b$ can still fit them all. Full-batch training, starting from $W = 0$.

In [ ]:
M_GROUPS = 7


def heavy_tailed(seed=0):
    """The problem of Kunstner et al. (Appendix A.2), identical to reel/research/adam_tail.py."""
    g = torch.Generator().manual_seed(1000 + seed)
    ys, gs, k = [], [], 0
    for i in range(1, M_GROUPS + 1):
        for _ in range(2 ** (i - 1)):
            ys += [k] * 2 ** (M_GROUPS - i + 1)
            gs += [i - 1] * 2 ** (M_GROUPS - i + 1)
            k += 1
    X = torch.rand(len(ys), (M_GROUPS + 1) * 2**M_GROUPS, generator=g)
    return X, torch.tensor(ys), torch.tensor(gs), k


def train_linear(opt_name, eta, steps, seed=0):
    """The mean loss of each frequency group after `steps` steps."""
    X, y, group, c = heavy_tailed(seed)
    W = torch.zeros(X.shape[1], c, requires_grad=True)
    b = torch.zeros(c, requires_grad=True)
    opt = {"gd": torch.optim.SGD([W, b], lr=eta), "adam": torch.optim.Adam([W, b], lr=eta), "sign": None}[opt_name]
    for _ in range(steps):
        W.grad = b.grad = None
        nn.functional.cross_entropy(X @ W + b, y).backward()
        if opt is None:  # sign descent: each parameter takes a step of η along the sign of its own gradient
            with torch.no_grad():
                W -= eta * W.grad.sign()
                b -= eta * b.grad.sign()
        else:
            opt.step()
    with torch.no_grad():
        per = nn.functional.cross_entropy(X @ W + b, y, reduction="none")
    return [per[group == k].mean().item() for k in range(M_GROUPS)]


X, y, group, n_cls = heavy_tailed()
print(
    f"{num(len(y), 0)} examples, {n_cls} classes, {num(X.shape[1], 0)} dimensions; "
    f"examples per group: {torch.bincount(group).tolist()}"
)
print(f"initial loss (W = 0) = ln {n_cls} = {num(math.log(n_cls), 3)}")

Each optimizer uses the learning rate the script chose for it by the paper's own rule: on a grid of powers
$10^{k/2}$, take the $\eta$ whose final loss, **worst of 3 seeds**, is smallest. The data are separable, so a huge
$\eta$ can send the loss up into the thousands and still bring it back close to 0; in that case the paper takes "the
next smaller step size", and the script states its rule explicitly: only the $\eta$ whose loss never exceeds the
initial level $\ln 127$ count.

### 🤔 Predict first
After 300 steps of gradient descent, which group has the highest loss: group 1 (the most common class) or group 7
(classes with only 2 examples each)? And with Adam?

In [ ]:
best = TAIL["heavy"]["best"]
live = {o: train_linear(o, best[o], 300) for o in ("gd", "sign", "adam")}
names = {"gd": "gradient descent", "sign": "sign descent", "adam": "Adam"}
print("loss per frequency group after 300 steps (group 1 = most common)")
for o, v in live.items():
    print(f"{names[o]:16} η = {sigfig(best[o], 2):>6}: " + "  ".join(num(x, 3) for x in v))
i300 = TAIL["check"].index(300)
for o, v in live.items():
    stored = TAIL["heavy"]["runs"][o][0][i300]
    assert all(abs(a - b) <= 2e-3 * max(1.0, b) for a, b in zip(v, stored)), f"{o}: differs from the stored result"

Gradient descent: the loss **rises with rarity** — group 1 is already down to 0.35, group 7 is still at 3.3, close
to the initial level of 4.84. Adam: every group is below 0.15, and the rare groups are even lower than the common
ones. Sign descent reverses the order completely: the rarest group is already at 0.009, while the most common one is
still at 0.88, higher even than with gradient descent. It isn't faster everywhere; it just doesn't leave the rare
classes behind.

Why? The gradient of the loss with respect to the weights of class $k$ is proportional to that class's frequency, so
rare classes receive small gradients. Gradient descent takes a step proportional to the gradient, and its $\eta$ is
limited by the most common class (Lesson 05: a step larger than $2/\lambda$ along the direction of largest curvature
$\lambda$ blows up). Adam and sign descent divide each gradient by its own magnitude, so every class takes steps of
about the same size, whether it is rare or common. The figure below shows the full 3 seeds, 500 steps.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)
steps_logged = TAIL["check"]
cmap = plt.get_cmap("viridis")
for ax, o in zip(axes, ("gd", "sign", "adam")):
    curves = np.array(TAIL["heavy"]["runs"][o])  # (seed, check, group)
    med = np.median(curves, 0)
    for k in range(M_GROUPS):
        ax.plot(steps_logged, np.maximum(med[:, k], 1e-4), color=cmap(k / (M_GROUPS - 1)),
                ls="-" if k % 2 == 0 else "--", lw=1.6, label=f"group {k + 1}")  # fmt: skip
    ax.set(xscale="log", yscale="log", title=f"{names[o]} (η = {sigfig(best[o], 2)})", xlabel="step")
    ax.set_ylim(1e-4, 10)
axes[0].set_ylabel("loss of the group (median of 3 seeds)")
axes[2].legend(fontsize=7, frameon=False, ncol=2, loc="lower left")
fmt_axes(fig)
plt.tight_layout()
plt.show()

Each panel is one optimizer; the horizontal axis is the number of steps, the vertical axis the loss of each group
(both axes on a log scale). The color goes from purple (group 1, the most common) to yellow (group 7, the rarest),
with solid and dashed lines alternating to tell neighboring groups apart. With gradient descent, the curves are
stacked in tiers: the rarer the group, the higher it sits. With sign descent and Adam, the order is reversed: the
rare groups come down first and go lowest.

**Control**: the same problem, but with 127 **balanced** classes (7 examples per class). If the gap comes from the
long tail, it must shrink when the tail is removed:

In [ ]:
for dist, label in (("heavy", "heavy-tailed"), ("uniform", "balanced")):
    finals = {o: [run[-1] for run in TAIL[dist]["runs"][o]] for o in ("gd", "sign", "adam")}
    tot = {o: [float(np.mean(f)) for f in fs] for o, fs in finals.items()}
    print(f"{label:13}: " + " · ".join(f"{names[o]} η {sigfig(TAIL[dist]['best'][o], 2)}: loss {sigfig(min(tot[o]))}–"
                                         f"{sigfig(max(tot[o]))}" for o in tot))  # fmt: skip

When the classes are balanced, gradient descent tolerates an $\eta$ 10 times larger (1 instead of 0.1), and its final
loss falls from about 0.97 to 0.028. Adam is still lower, but the absolute gap almost disappears. What makes gradient
descent slow is not that the problem is hard, but that **a common class forces it to take small steps, while the rare
classes need large ones**.

**Verdict: holds.** On the paper's own problem, gradient descent falls behind precisely on the rare classes, while
Adam and sign descent don't; the balanced control shrinks the gap considerably. Limitation: this is a linear model on
random data; the paper also uses a frozen transformer on real text, which this notebook doesn't rerun.

## 4 · The edge of stability: the curvature stops by itself at $2/\eta$

**Claim** (Cohen et al., 2021): Lesson 05 says that gradient descent with step size $\eta$ is stable only when the
**sharpness** $S$, the largest eigenvalue of the Hessian matrix of the loss, is smaller than $2/\eta$. In neural
networks trained full-batch, one observes that $S$ first **rises steadily** ("progressive sharpening") up to exactly
$2/\eta$, then **hovers there** — while the loss still falls, though no longer at every step. The network brings
itself to the edge of the stable region. Cohen et al. (2024) later proposed "central flows", an equation that predicts
the average trajectory in this regime; the notebook only checks the phenomenon, not that theory.

The experiment: an MLP 784 → 128 → 128 → 10 with tanh, the squared loss $\tfrac12\lVert f(x) - y\rVert^2$ (as in
Cohen's main experiment; with cross-entropy, the sharpness gradually falls toward the end), 1,000 MNIST images,
full-batch gradient descent. $S$ is measured by **power iteration** on Hessian–vector products: there is no need to
build the Hessian (more than a hundred thousand × a hundred thousand entries), only to ask "what is the Hessian times
the vector $v$", which PyTorch computes with two backpropagation passes.

In [ ]:
def mnist_first(n=1000):
    with gzip.open(DATA / "train-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)[:n]
    with gzip.open(DATA / "train-labels-idx1-ubyte.gz") as f:
        lab = np.frombuffer(f.read(), np.uint8, offset=8)[:n]
    return torch.from_numpy(x.astype(np.float32) / 255), nn.functional.one_hot(
        torch.from_numpy(lab.astype(np.int64)), 10
    ).float()


Xm, Ym = mnist_first()


def mlp_tanh(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(784, 128), nn.Tanh(), nn.Linear(128, 128), nn.Tanh(), nn.Linear(128, 10))


def mse(m):
    return ((m(Xm) - Ym) ** 2).sum(1).mean() / 2


def sharpness(m, v=None, iters=20):
    """The largest eigenvalue of the Hessian by power iteration: v ← Hv / ‖Hv‖, repeated. Returns (S, v)."""
    ps = list(m.parameters())
    g = torch.autograd.grad(mse(m), ps, create_graph=True)
    v = [torch.randn_like(p) for p in ps] if v is None else v
    for _ in range(iters):
        norm = torch.sqrt(sum((a * a).sum() for a in v))
        v = [a / norm for a in v]
        hv = torch.autograd.grad(g, ps, v, retain_graph=True)
        lam = sum((a * b).sum() for a, b in zip(hv, v)).item()
        v = [h.detach() for h in hv]
    return lam, v


def gd_run(eta, steps, seed=0, every=25):
    """Full-batch gradient descent; S every `every` steps (warm-started from the previous vector), the loss every step."""
    m = mlp_tanh(seed)
    opt = torch.optim.SGD(m.parameters(), lr=eta)
    loss_curve, sharp, v = [], [], None
    for s in range(steps + 1):
        if s % every == 0:
            lam, v = sharpness(m, v)
            sharp.append(lam)
        opt.zero_grad()
        loss = mse(m)
        loss_curve.append(loss.item())
        if s < steps:
            loss.backward()
            opt.step()
    return loss_curve, sharp


n_params = sum(p.numel() for p in mlp_tanh(0).parameters())
print(f"{n_params:,} parameters")

### 🤔 Predict first
With $\eta = 0.1$, the classical stability limit is $2/\eta = 20$. At initialization, $S \approx 9.5$. After a few
hundred steps, will $S$ (a) still be around 9.5, (b) go well past 20 and the network blow up, or (c) stop around 20?

In [ ]:
loss_live, sharp_live = gd_run(0.1, 400)
print("step   S      loss")
for s in (0, 100, 200, 300, 400):
    print(f"{s:4}  {num(sharp_live[s // 25], 1):>5}  {num(loss_live[s], 4)}")
stored = EOS["runs"]["0.1"][0]
assert abs(sharp_live[300 // 25] - stored["sharp"][300 // 25]) < 1e-2 * stored["sharp"][300 // 25], (
    "differs from the stored result"
)
assert abs(loss_live[400] - stored["loss"][400 // EOS["lossEvery"]]) < 1e-3, "differs from the stored result"
ups = sum(b > a for a, b in pairwise(loss_live))
print(f"steps at which the loss went up: {ups}/400")

$S$ rises from 9.5 to a little above 20, then oscillates around it: (c) is right. And from then on the loss no longer
falls steadily; it goes up at some steps, though the overall trend is still downward. The full results: 4 learning
rates × 3 seeds × 3,000 steps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
steps_s = np.arange(0, EOS["steps"] + 1, EOS["every"])
steps_l = np.arange(0, EOS["steps"] + 1, EOS["lossEvery"])
colors = {"0.02": GREY, "0.05": GREEN, "0.1": BLUE, "0.2": ORANGE}
styles = {"0.02": ":", "0.05": "-.", "0.1": "-", "0.2": "--"}
for e, runs in EOS["runs"].items():
    eta = float(e)
    S = np.median([r_["sharp"] for r_ in runs], 0)
    L = np.median([r_["loss"] for r_ in runs], 0)
    axes[0].plot(steps_s, S, color=colors[e], ls=styles[e], lw=1.8, label=f"η = {num(eta, 2)}")
    axes[0].axhline(2 / eta, color=colors[e], lw=0.8, alpha=0.6)
    axes[1].plot(steps_l, L, color=colors[e], ls=styles[e], lw=1.4, label=f"η = {num(eta, 2)}")
axes[0].set(xlabel="step", ylabel="sharpness S (median of 3 seeds)", ylim=(0, 110), title="S and the 2/η lines")
axes[1].set(xlabel="step", ylabel="loss (median of 3 seeds)", yscale="log", title="Loss")
axes[0].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()

In [ ]:
for e, runs in EOS["runs"].items():
    eta = float(e)
    end = [r_["sharp"][-1] for r_ in runs]
    late = [float(np.mean(r_["sharp"][len(r_["sharp"]) // 2 :])) for r_ in runs]
    final = [r_["loss"][-1] for r_ in runs]
    ups = [sum(b > a for a, b in pairwise(r_["loss"])) for r_ in runs]
    print(f"η = {num(eta, 2):>4}: 2/η = {num(2 / eta, 0):>3} · mean S over the second half {num(min(late), 1)}–{num(max(late), 1)}, "
          f"at the last step {num(min(end), 1)}–{num(max(end), 1)} · "
          f"final loss {num(min(final), 3)}–{num(max(final), 3)} · rises (checked every 5 steps) {min(ups)}–{max(ups)}/600")  # fmt: skip

On the left, each curve is $S$ for one learning rate (each with its own color and line style), and each thin
horizontal line of the same color is its $2/\eta$. With $\eta = 0.05$, 0.1 and 0.2, $S$ climbs and then stays right
at its line (mean over the second half 40.3–40.7, 20.7–20.9 and 10.5–10.7, against 40, 20 and 10). With
$\eta = 0.02$, the line is at 100 and $S$ is still rising when the 3,000 steps run out, far from the line: the network
is still in the classical regime, and the loss falls at every measurement point. On the right: the larger the
learning rate, the **lower** the final loss — running at the edge of stability is not a failure, it is the normal
operating regime of gradient descent with a large $\eta$.

**Verdict: holds.** The sharpness rises by itself to $2/\eta$ and stops there, at all three large-enough learning
rates and in all 3 seeds. Lesson 05's rule "choose $\eta < 2/S$" still holds for each step, but in a neural network
$S$ isn't fixed: it changes with the very $\eta$ we choose.

## 5 · Grokking and softmax collapse

**Claim.** On modular arithmetic, a network can reach 100% on the training set very early while the test set stays at
chance level, and then, much later, suddenly generalize: **grokking** (Power et al., 2022). It usually happens only
with weight decay. Prieto et al. (2025) explain why grokking often doesn't come without weight decay:
1. Once every training example is classified correctly, the cheapest way to lower the cross-entropy is to **scale up
   the logits** in the direction they already point: the update becomes almost parallel to the weights themselves
   ("naïve loss minimization").
2. The logits grow so large that a float32 softmax rounds the probability of the correct class to **exactly 1**: the
   loss and the gradient of that example are exactly 0, and learning stops. They call this **softmax collapse**.

Their two fixes: **StableMax**, a softmax in which $s(x)$ grows linearly instead of exponentially, and **⊥Grad**,
which removes the component of the gradient parallel to the weights before each step. The experiment uses exactly the
setup in their source code: $(a + b) \bmod 113$, 40% of the pairs for training, an MLP 226 → 200 → 200 → 113 without
biases, full-batch AdamW with $\varepsilon = 10^{-25}$.

Each experiment below runs 3 seeds; the seed determines the 40/60 split, the initial weights and the shuffling order
at each epoch.

In [ ]:
P_MOD = 113


def mod_data(seed):
    """(a + b) mod 113, every pair, 40% for training; identical to reel/research/grok.py."""
    g = torch.Generator().manual_seed(seed)
    a, b = torch.meshgrid(torch.arange(P_MOD), torch.arange(P_MOD), indexing="ij")
    a, b = a.flatten(), b.flatten()
    y = (a + b) % P_MOD
    X = torch.cat([nn.functional.one_hot(a, P_MOD), nn.functional.one_hot(b, P_MOD)], 1).float()
    perm = torch.randperm(P_MOD * P_MOD, generator=g)
    n = int(0.4 * P_MOD * P_MOD)
    return X[perm[:n]], y[perm[:n]], X[perm[n:]], y[perm[n:]]


def mod_mlp(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2 * P_MOD, 200, bias=False), nn.ReLU(), nn.Linear(200, 200, bias=False), nn.ReLU(),
                         nn.Linear(200, P_MOD, bias=False))  # fmt: skip


def orthogonalise(params):
    """⊥Grad (the authors' code): remove the component of the gradient parallel to the weights, then keep the
    gradient's original length."""
    with torch.no_grad():
        for q in params:
            w, g = q.view(-1), q.grad.view(-1)
            go = g - (w @ g) / (w @ w + 1e-30) * w
            q.grad.copy_((go * (g.norm() / (go.norm() + 1e-30))).view_as(q.grad))


def grok_run(ortho, epochs=200, seed=0, lr=0.01):
    Xtr, ytr, Xte, yte = mod_data(seed)
    m = mod_mlp(seed)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, betas=(0.9, 0.99), eps=1e-25, weight_decay=0.0)
    torch.manual_seed(10_000 + seed)  # the shuffling order at each epoch
    log = []
    for e in range(epochs + 1):
        if e % 50 == 0:
            with torch.no_grad():
                log.append((e, (m(Xtr).argmax(1) == ytr).float().mean().item(),
                            (m(Xte).argmax(1) == yte).float().mean().item()))  # fmt: skip
        if e == epochs:
            break
        perm = torch.randperm(len(Xtr))
        loss = nn.functional.cross_entropy(m(Xtr[perm]), ytr[perm])
        opt.zero_grad()
        loss.backward()
        if ortho:
            orthogonalise(m.parameters())
        opt.step()
    return log


Xtr, ytr, Xte, yte = mod_data(0)
print(f"{num(len(ytr), 0)} pairs for training, {num(len(yte), 0)} pairs for testing")

### 🤔 Predict first
Train for 200 epochs, without weight decay, with plain AdamW and with AdamW plus ⊥Grad. After 200 epochs, which
network is right on the test set: neither, only ⊥Grad, or both?

In [ ]:
live = {name: grok_run(ortho) for name, ortho in (("plain AdamW", False), ("AdamW + ⊥Grad", True))}
for name, log in live.items():
    print(f"{name:14}: " + " · ".join(f"epoch {e}: training {pct(a, 0)}, test {pct(b, 1)}" for e, a, b in log[1::2]))
for (name, log), key in zip(live.items(), ("softmax-lr0.01", "ortho-lr0.01")):
    stored = GROK["live"]["runs"][key]
    assert abs(log[-1][2] - stored["testAcc"][-1]) < 0.05, f"{name}: differs from the stored result"

Plain AdamW memorizes the training set by epoch 50 but does no better than chance on the test set. With ⊥Grad, the
network learns the training set more slowly, then at epoch 150 gets almost every **unseen** pair right. The full
results, 3 seeds:

In [ ]:
LABEL = {
    "softmax-lr0.005": "softmax, float32, η 0.005", "softmax-lr0.01": "softmax, float32, η 0.01",
    "softmax-f64": "softmax, float64", "stablemax": "StableMax (40,000 epochs)",
    "stablemax-80k": "StableMax (80,000 epochs)", "ortho-lr0.005": "⊥Grad, η 0.005", "ortho-lr0.01": "⊥Grad, η 0.01",
    "wd1": "weight decay 1",
}  # fmt: skip


def epochs_text(es):
    """[150, 150, None] → "150, 150, never" (never = not at any point during the run)."""
    return ", ".join("never" if e is None else num(e, 0) for e in es)


for a, runs in GROK["runs"].items():
    grok_at = [r_["grokEpoch"] for r_ in runs]
    first_zero = [next((e for e, z in zip(r_["epoch"], r_["zero"]) if z > 0.01), None) for r_ in runs]
    print(f"{LABEL[a]:26} final test {pct(min(r_['testAcc'][-1] for r_ in runs), 1)}–"
          f"{pct(max(r_['testAcc'][-1] for r_ in runs), 1)} · reaches 95% at epoch {epochs_text(grok_at)} · "
          f"loss exactly 0 on at most {pct(max(max(r_['zero']) for r_ in runs), 0)} of examples "
          f"(from epoch {epochs_text(first_zero)}) · max cos "
          f"{num(max(max(c for c in r_['cos'] if c is not None) for r_ in runs), 2)}")  # fmt: skip

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
show = [("softmax-lr0.01", GREY, ":"), ("softmax-f64", ORANGE, "--"), ("wd1", GREEN, "-."), ("ortho-lr0.01", BLUE, "-")]
show.append(("stablemax-80k" if "stablemax-80k" in GROK["runs"] else "stablemax", "#7a4fb5", (0, (5, 1, 1, 1))))
for a, col, ls in show:
    runs = GROK["runs"][a]
    ep = np.array(runs[0]["epoch"], float)
    ep[0] = 1  # log axis: epoch 0 is drawn at 1
    axes[0].plot(ep, np.median([r_["testAcc"] for r_ in runs], 0), color=col, ls=ls, lw=1.8, label=LABEL[a])
    if a.startswith("softmax") or a == "wd1":
        axes[1].plot(ep, np.median([r_["zero"] for r_ in runs], 0), color=col, ls=ls, lw=1.8, label=LABEL[a])
axes[0].set(xscale="log", xlabel="epoch", ylabel="test accuracy (median of 3 seeds)", title="Generalization")
axes[1].set(xscale="log", xlabel="epoch", ylabel="training examples with a loss of exactly 0",
            title="Softmax collapse")  # fmt: skip
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[0].legend(frameon=False, fontsize=8)
axes[1].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()

On the left is the test accuracy by epoch (horizontal axis on a log scale), one color and one line style per way of
training. On the right is the share of training examples whose loss is **exactly 0** in floating point: the sign of
softmax collapse.

- **Plain softmax, float32:** memorizes the training set within 50–100 epochs, then the update becomes almost
  parallel to the weights themselves ("max cos" in the cell above: the cosine between the raw −gradient and the
  weights; close to 1 means the update mostly scales up the logits). From about epoch 1,600, examples start to have
  a loss of exactly 0; that share fluctuates strongly, at times reaching 89–97% of the training set. On the test set,
  the network never does better than chance.
- **Float64:** the collapse comes much later (from about epoch 5,500) and is smaller, but there is still no grokking
  within 6,000 epochs: with 40% of the data, the paper also finds that the collapse arrives before the test accuracy
  has had time to move, even with float64.
- **⊥Grad:** its raw gradient is also almost parallel to the weights (cos up to 0.84–0.94, measured before the
  projection), but that component is removed before each step. No example ever has a loss of 0, and it groks at
  epoch 150–200 in all 3 seeds, without any weight decay.
- **Weight decay:** groks at epoch 1,000. As many as about 30% of the examples also reach a loss of 0 midway, but
  weight decay pulls the logits back, so the rest of the training set still provides a gradient.
- **StableMax:** also groks, but much more slowly than ⊥Grad. After 40,000 epochs, the test accuracy is only at
  12–59%; after 80,000 epochs (the length in the authors' code), 2 of the 3 seeds pass 95% (at epochs 60,000 and
  66,400), and all three end at 90.6–98.8%. No example has a loss of 0: StableMax grows linearly, not exponentially,
  so nothing gets rounded off.

**Verdict: holds, with one condition.** Without weight decay, a float32 softmax collapses and doesn't grok; both of
the paper's fixes grok without weight decay: ⊥Grad very quickly, StableMax very slowly. But these results are
sensitive to the hyperparameters: the first attempt while preparing this notebook (modulo 59, with biases, Adam's
default $\varepsilon = 10^{-8}$) found no help from either ⊥Grad or StableMax. Only with exactly the setup in the
authors' source code did the phenomenon show up clearly.

## Summary
- **Adam and rare classes — holds.** On the problem of Kunstner et al., gradient descent leaves the rare classes at a
  high loss (the rarest group at 3.3 after 300 steps), while Adam brings every group below 0.15; when the classes are
  balanced, the gap shrinks considerably.
- **The edge of stability — holds.** The sharpness rises by itself to $2/\eta$ and stops there with $\eta$ of 0.05,
  0.1 and 0.2, in all 3 seeds; the loss still falls, though unevenly.
- **Grokking and softmax collapse — holds, but sensitive to the setup.** A float32 softmax without weight decay
  collapses and never generalizes; ⊥Grad groks after 150–200 epochs, weight decay after 1,000, StableMax after
  60,000–66,400 (2 of 3 seeds, within 80,000 epochs).
- All three results appear at a scale small enough to run on a CPU. That doesn't prove they hold for large models,
  but it shows that the mechanisms behind them don't need a large model to appear.

## Review questions
1. Why is the gradient of a class's weights proportional to that class's frequency, and where does that make
   gradient descent slow?
2. Sign descent doesn't use the magnitude of the gradient. Why does it still beat gradient descent on this problem?
3. At the edge of stability, the loss doesn't fall at every step. Why doesn't that contradict the fact that it still
   falls in the long run?
4. Why don't we see the edge of stability with $\eta = 0.02$ within 3,000 steps?
5. What is "softmax collapse", and why does float64 make it come later?
6. Which part of the gradient does ⊥Grad remove, and why does removing that part help the network generalize?

<details><summary>Hints and answers</summary>

1. The loss is an average over the examples, and only the examples of class $k$ push the logit of class $k$ up
   strongly; a class with few examples contributes little in total. Gradient descent takes steps proportional to the
   gradient, and $\eta$ is already limited by the most common class, so the rare classes move very slowly.
2. It takes a step of the same size for every parameter, so the weights of the rare classes move as fast as those of
   the common classes.
3. The network oscillates back and forth along the sharpest direction, but still makes progress along the other
   directions; Cohen et al. (2024) describe that average trajectory with "central flows".
4. $2/\eta = 100$, while the sharpness is still rising steadily and is still far from 100 when the 3,000 steps run
   out: the network hasn't reached the edge yet.
5. The logits grow so large that float32 rounds the probability of the correct class to exactly 1, so the loss and
   the gradient of that example are exactly 0. Float64 has more digits, so the logits have to grow even larger first.
6. The component parallel to the weights, that is, the part that only scales up the logits without changing the
   predictions. Without it, every step has to change the direction of the weights, that is, change how the network
   computes, and the loss can't fall just by scaling up.
</details>

## Further reading
- F. Kunstner et al. (2024), [Heavy-Tailed Class Imbalance and Why Adam Outperforms Gradient Descent on Language
  Models](https://arxiv.org/abs/2402.19449); F. Kunstner & F. Bach (2025), [Scaling Laws for Gradient Descent and Sign
  Descent for Linear Bigram Models under Zipf's Law](https://arxiv.org/abs/2505.19227); R. Yadav et al. (2025),
  [Provable Benefit of SignGD](https://arxiv.org/abs/2512.00763).
- J. Cohen et al. (2021), [Gradient Descent on Neural Networks Typically Occurs at the Edge of
  Stability](https://arxiv.org/abs/2103.00065); J. Cohen et al. (2024), [Understanding Optimization in Deep
  Learning with Central Flows](https://arxiv.org/abs/2410.24206).
- L. Prieto et al. (2025), [Grokking at the Edge of Numerical Stability](https://arxiv.org/abs/2501.04697) and their
  source code (the setup that section 5 uses); A. Power et al. (2022),
  [Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets](https://arxiv.org/abs/2201.02177).

**Back to:** [Lesson 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/).

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
      f"matplotlib {matplotlib.__version__}")  # fmt: skip